# 🔬 Lab Module 28 — Agent Governance & Ethical AI Frameworks

**FinTech Corp LoanBot v4.0 — AI Governance Implementation**

Sau khi NHNN phát công văn về rural bias, FinTech Corp cần xây dựng toàn bộ governance framework từ đầu.
Lab này hướng dẫn từng bước: Policy Engine → Fairness Auditor → XAI Explainer → Compliance Reporter → Ethics Escalation.

---

## Mục tiêu Lab
1. Build **PolicyEngine** — enforce governance charter tự động
2. Implement **FairnessAuditor** — detect disparate impact theo 80% rule
3. Create **LoanBotExplainer** — counterfactual + NLE explanations
4. Generate **Compliance Evidence Package** cho NHNN audit
5. Wire **EthicsEscalationEngine** — auto-detect và escalate ethics violations
6. Build **GovernancePipeline** — integrate tất cả thành 1 production workflow

In [ ]:
# Section 0: Setup
!pip install anthropic --quiet

import os
import json
import random
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
from enum import Enum
from datetime import datetime, timedelta

print('✅ Dependencies loaded')
print('📋 Module 28 Lab: AI Governance & Ethical AI Frameworks')

## Section 1: Data Simulation — LoanBot Decision History

Simulate 3000 loan decisions với demographic attributes và outcomes.
Real-world pattern: rural borrowers có lower approval (bias) và younger borrowers bị disadvantaged.

In [ ]:
# Section 1: Simulate LoanBot Decision Dataset

@dataclass
class LoanCase:
    case_id: str
    credit_score: int
    dti_ratio: float
    employment_months: int
    loan_amount: int
    region: str          # 'urban' / 'rural'
    age_group: str       # '18-30' / '30-50' / '50+'
    gender: str          # 'M' / 'F'
    agent_decision: str  # 'APPROVE' / 'REVIEW' / 'REJECT'
    confidence: float
    outcome_30d: Optional[str] = None  # 'REPAID' / 'DEFAULTED' / None

def simulate_loanbot_decision(case: LoanCase, biased: bool = True) -> str:
    """Simulate LoanBot v3.9 decision — with bias if biased=True."""
    score = case.credit_score
    # Clear thresholds
    if score < 500 or case.dti_ratio > 0.55:
        return 'REJECT'
    if score >= 720 and case.dti_ratio <= 0.35 and case.employment_months >= 24:
        return 'APPROVE'
    # Biased model: rural penalty + young penalty
    adjusted_score = score
    if biased:
        if case.region == 'rural':     adjusted_score -= 35  # proxy discrimination!
        if case.age_group == '18-30':  adjusted_score -= 20
    if case.employment_months < 9:     adjusted_score -= 30
    if adjusted_score >= 640 and case.dti_ratio <= 0.42:
        return 'APPROVE'
    elif adjusted_score >= 580:
        return 'REVIEW'
    return 'REJECT'

def generate_dataset(n: int = 3000, biased: bool = True) -> List[LoanCase]:
    random.seed(42)
    cases = []
    regions     = ['urban'] * 70 + ['rural'] * 30
    age_groups  = ['18-30'] * 25 + ['30-50'] * 55 + ['50+'] * 20
    genders     = ['M'] * 55 + ['F'] * 45
    for i in range(n):
        region    = random.choice(regions)
        age_group = random.choice(age_groups)
        # Rural borrowers tend to have lower employment stability
        emp_months = random.randint(3, 48) if region == 'rural' else random.randint(6, 84)
        score      = random.randint(480, 780)
        dti        = round(random.uniform(0.20, 0.58), 2)
        case = LoanCase(
            case_id=f'FC-2026-{i+1:04d}',
            credit_score=score, dti_ratio=dti, employment_months=emp_months,
            loan_amount=random.choice([100, 200, 400, 600, 1000, 2000]) * 1_000_000,
            region=region, age_group=age_group, gender=random.choice(genders),
            agent_decision='PENDING', confidence=round(random.uniform(0.60, 0.95), 2)
        )
        case.agent_decision = simulate_loanbot_decision(case, biased=biased)
        # Simulate 30-day outcomes for approved cases
        if case.agent_decision == 'APPROVE':
            default_prob = 0.08 + (0.42 - case.dti_ratio) * 0.1 if case.dti_ratio < 0.42 else 0.18
            case.outcome_30d = 'DEFAULTED' if random.random() < default_prob else 'REPAID'
        cases.append(case)
    return cases

dataset = generate_dataset(3000, biased=True)

# Quick stats
urban_cases  = [c for c in dataset if c.region == 'urban']
rural_cases  = [c for c in dataset if c.region == 'rural']
urban_ar = sum(1 for c in urban_cases if c.agent_decision == 'APPROVE') / len(urban_cases)
rural_ar = sum(1 for c in rural_cases if c.agent_decision == 'APPROVE') / len(rural_cases)

print(f'📊 Dataset: {len(dataset)} cases')
print(f'   Urban: {len(urban_cases)} cases, Approval Rate: {urban_ar:.1%}')
print(f'   Rural: {len(rural_cases)} cases, Approval Rate: {rural_ar:.1%}')
print(f'   Disparity Ratio: {rural_ar/urban_ar:.3f} (80% rule threshold: 0.800)')
if rural_ar / urban_ar < 0.80:
    print(f'   ⚠️  BIAS DETECTED: Disparity {rural_ar/urban_ar:.3f} < 0.80 minimum!')

## Section 2: Governance Policy Engine

Build PolicyEngine từ FinTech Corp's Governance Charter.
Tự động check tất cả policies và generate violation alerts.

In [ ]:
# Section 2: Governance Policy Engine

class RiskLevel(Enum):
    LOW      = 1
    MEDIUM   = 2
    HIGH     = 3
    CRITICAL = 4

@dataclass
class GovernancePolicy:
    policy_id: str
    name: str
    threshold: float
    metric: str
    risk_level: RiskLevel
    action_required: str
    escalation_path: List[str]
    review_frequency_days: int
    compare: str = 'lt'  # 'lt' = value must be < threshold to violate; 'gt' = must be >

LOANBOT_CHARTER = [
    GovernancePolicy('GP-001', 'Demographic Parity (Rural/Urban)', 0.80,
        'approval_rate_ratio_rural', RiskLevel.CRITICAL,
        'Immediate halt of rural segment + manual review',
        ['AI Engineer', 'Risk Manager', 'CRO', 'NHNN'], 7, compare='lt'),
    GovernancePolicy('GP-002', 'Demographic Parity (Young/Reference)', 0.80,
        'approval_rate_ratio_young', RiskLevel.CRITICAL,
        'Review age-related features in model',
        ['AI Engineer', 'Risk Manager', 'CRO'], 14, compare='lt'),
    GovernancePolicy('GP-003', 'Type I Error Rate', 0.15,
        'type_i_error_rate', RiskLevel.HIGH,
        'Trigger self-improvement cycle within 30 days',
        ['AI Engineer', 'Risk Manager'], 30, compare='gt'),
    GovernancePolicy('GP-004', 'Explainability Coverage', 1.00,
        'explanation_coverage', RiskLevel.HIGH,
        'Generate explanations for all missing cases immediately',
        ['AI Engineer'], 1, compare='lt'),
    GovernancePolicy('GP-005', 'Human Override Rate', 0.05,
        'human_override_rate', RiskLevel.MEDIUM,
        'Review agent confidence calibration',
        ['AI Product Manager'], 30, compare='gt'),
]

def compute_metrics(cases: List[LoanCase]) -> Dict[str, float]:
    """Compute governance metrics from decision dataset."""
    urban = [c for c in cases if c.region == 'urban']
    rural = [c for c in cases if c.region == 'rural']
    young = [c for c in cases if c.age_group == '18-30']
    ref30 = [c for c in cases if c.age_group == '30-50']

    def ar(lst): return sum(1 for c in lst if c.agent_decision == 'APPROVE') / len(lst) if lst else 0

    # Type I error (approved + defaulted) / all approved with outcome
    approved_with_outcome = [c for c in cases if c.agent_decision == 'APPROVE' and c.outcome_30d]
    type_i = sum(1 for c in approved_with_outcome if c.outcome_30d == 'DEFAULTED') / len(approved_with_outcome) if approved_with_outcome else 0

    return {
        'approval_rate_ratio_rural': ar(rural) / ar(urban) if ar(urban) > 0 else 0,
        'approval_rate_ratio_young': ar(young) / ar(ref30) if ar(ref30) > 0 else 0,
        'type_i_error_rate': type_i,
        'explanation_coverage': 0.97,   # Simulated: 3% missing explanations
        'human_override_rate': 0.03,    # Simulated: 3% human overrides
    }

def run_policy_engine(cases: List[LoanCase]) -> List[dict]:
    metrics = compute_metrics(cases)
    violations = []
    print('\n📋 GOVERNANCE POLICY ENGINE REPORT')
    print('=' * 60)
    for policy in LOANBOT_CHARTER:
        value = metrics.get(policy.metric, None)
        if value is None: continue
        violated = (policy.compare == 'lt' and value < policy.threshold) or \
                   (policy.compare == 'gt' and value > policy.threshold)
        status = '❌ VIOLATION' if violated else '✅ PASS'
        direction = '↓ below' if policy.compare == 'lt' else '↑ above'
        print(f'{status} | {policy.policy_id}: {policy.name}')
        print(f'       Value: {value:.3f} | Threshold: {direction} {policy.threshold}')
        if violated:
            print(f'       Risk: {policy.risk_level.name} | Escalate to: {policy.escalation_path[-1]}')
            print(f'       Action: {policy.action_required}')
            violations.append({'policy_id': policy.policy_id, 'value': value,
                                'risk': policy.risk_level.name, 'escalate_to': policy.escalation_path[-1]})
        print()
    print(f'Summary: {len(violations)}/{len(LOANBOT_CHARTER)} policies violated')
    return violations

violations = run_policy_engine(dataset)

## Section 3: Fairness Auditor — Full Demographic Analysis

Compute Demographic Parity, Equalized Odds, và Calibration across all protected attributes.

In [ ]:
# Section 3: Full Fairness Audit

@dataclass
class FairnessReport:
    group_name: str
    n_cases: int
    approval_rate: float
    disparity_ratio: float
    tpr: float       # True Positive Rate among approved
    fpr: float       # False Positive Rate (approvals that defaulted)
    passes_80_rule: bool

def audit_group(cases: List[LoanCase], reference_cases: List[LoanCase]) -> dict:
    """Compute fairness metrics for group vs reference."""
    def ar(lst): return sum(1 for c in lst if c.agent_decision == 'APPROVE') / len(lst) if lst else 0
    def tpr(lst):
        approved = [c for c in lst if c.agent_decision == 'APPROVE' and c.outcome_30d]
        if not approved: return 0, 0
        tp = sum(1 for c in approved if c.outcome_30d == 'REPAID')
        fp = sum(1 for c in approved if c.outcome_30d == 'DEFAULTED')
        return tp / len(approved), fp / len(approved)
    g_ar, r_ar = ar(cases), ar(reference_cases)
    g_tpr, g_fpr = tpr(cases)
    r_tpr, r_fpr = tpr(reference_cases)
    ratio = g_ar / r_ar if r_ar > 0 else 0
    return {'n': len(cases), 'ar': g_ar, 'ratio': ratio, 'tpr': g_tpr, 'fpr': g_fpr,
            'tpr_gap': abs(g_tpr - r_tpr), 'passes_80': ratio >= 0.80}

def full_fairness_audit(cases: List[LoanCase]):
    urban = [c for c in cases if c.region == 'urban']
    rural = [c for c in cases if c.region == 'rural']
    male  = [c for c in cases if c.gender == 'M']
    female= [c for c in cases if c.gender == 'F']
    young = [c for c in cases if c.age_group == '18-30']
    mid   = [c for c in cases if c.age_group == '30-50']   # reference
    senior= [c for c in cases if c.age_group == '50+']

    groups = [
        ('Urban (ref)', urban, urban),
        ('Rural',       rural, urban),
        ('Male',        male,  male),
        ('Female',      female,male),
        ('Age 18-30',   young, mid),
        ('Age 30-50 (ref)', mid, mid),
        ('Age 50+',     senior, mid),
    ]

    print('\n🔍 FULL FAIRNESS AUDIT REPORT')
    print('=' * 85)
    print(f'{"Group":<20} {"N":>5} {"Approval%":>10} {"Ratio":>8} {"80%Rule":>8} {"TPR":>6} {"FPR":>6} {"EO Gap":>8}')
    print('-' * 85)
    violations = []
    for name, grp, ref in groups:
        r = audit_group(grp, ref)
        rule_str = 'PASS' if r['passes_80'] else 'FAIL ❌'
        if not r['passes_80'] and name != 'Urban (ref)' and name != 'Male' and name != 'Age 30-50 (ref)':
            violations.append(name)
        print(f"{name:<20} {r['n']:>5} {r['ar']:>10.1%} {r['ratio']:>8.3f} {rule_str:>8} {r['tpr']:>6.1%} {r['fpr']:>6.1%} {r['tpr_gap']:>8.1%}")
    print('=' * 85)
    print(f'\n⚠️  Fairness violations: {violations if violations else "None detected"}')
    return violations

audit_violations = full_fairness_audit(dataset)

## Section 4: XAI — Counterfactual Explanation Generator

Implement counterfactual explanation: minimum changes to flip decision to APPROVE.
Test trên FC-2024-004 (score 645, DTI 0.42, emp 8 months).

In [ ]:
# Section 4: XAI Counterfactual Explainer

@dataclass
class Counterfactual:
    feature: str
    current_value: object
    required_value: object
    change_description: str
    feasibility: str    # 'EASY' / 'MODERATE' / 'HARD'
    impact_score: float  # 0-1, how much this alone can flip decision

@dataclass
class ThresholdConfig:
    min_score: int = 620
    max_dti: float = 0.42
    min_emp_months: int = 12

class LoanBotExplainer:
    FEASIBILITY_ORDER = {'EASY': 0, 'MODERATE': 1, 'HARD': 2}

    def __init__(self, cfg: ThresholdConfig = None):
        self.cfg = cfg or ThresholdConfig()

    def feature_importance(self, app: dict) -> List[dict]:
        """LIME-style feature attribution — simplified simulation."""
        contributions = []
        # Employment tenure — main driver for borderline cases
        if app['employment_months'] < self.cfg.min_emp_months:
            deficit = self.cfg.min_emp_months - app['employment_months']
            contributions.append({'feature': 'employment_months', 'direction': 'negative',
                                   'impact': -0.30 * deficit / self.cfg.min_emp_months,
                                   'label': f"Thời gian làm việc {app['employment_months']}mo (cần {self.cfg.min_emp_months}mo)"})
        # DTI
        if app['dti_ratio'] > self.cfg.max_dti:
            contributions.append({'feature': 'dti_ratio', 'direction': 'negative',
                                   'impact': -0.18, 'label': f"DTI {app['dti_ratio']:.2f} (max {self.cfg.max_dti:.2f})"})
        # Credit score — positive
        score_pct = (app['credit_score'] - 500) / (750 - 500)
        contributions.append({'feature': 'credit_score', 'direction': 'positive',
                               'impact': 0.12 * score_pct, 'label': f"Score {app['credit_score']} (tốt)"})
        return sorted(contributions, key=lambda x: x['impact'])

    def counterfactual_explanation(self, app: dict) -> List[Counterfactual]:
        """Find minimum changes to flip decision to APPROVE."""
        cfs = []
        # Employment change
        if app['employment_months'] < self.cfg.min_emp_months:
            months_needed = self.cfg.min_emp_months - app['employment_months']
            cfs.append(Counterfactual(
                feature='employment_months',
                current_value=app['employment_months'],
                required_value=self.cfg.min_emp_months,
                change_description=f"Tăng thời gian làm việc từ {app['employment_months']} lên {self.cfg.min_emp_months} tháng (cần chờ ~{months_needed} tháng)",
                feasibility='MODERATE',
                impact_score=0.85
            ))
        # DTI change
        if app['dti_ratio'] > self.cfg.max_dti:
            cfs.append(Counterfactual(
                feature='dti_ratio', current_value=app['dti_ratio'],
                required_value=self.cfg.max_dti,
                change_description=f"Giảm DTI từ {app['dti_ratio']:.2f} xuống {self.cfg.max_dti:.2f} (trả bớt nợ hiện tại)",
                feasibility='HARD', impact_score=0.60
            ))
        # Collateral — alternative path
        cfs.append(Counterfactual(
            feature='collateral', current_value=0, required_value=1,
            change_description='Cung cấp tài sản đảm bảo (có thể bypass yêu cầu thời gian công tác theo TT39/2016 Điều 9)',
            feasibility='EASY', impact_score=0.90
        ))
        return sorted(cfs, key=lambda x: self.FEASIBILITY_ORDER[x.feasibility])

    def generate_customer_letter(self, app: dict, decision: str, cfs: List[Counterfactual]) -> str:
        """Generate Vietnamese explanation letter without API call (template-based)."""
        cf_lines = '\n'.join([f'  {'✓' if c.feasibility == 'EASY' else '○'} {c.change_description}' for c in cfs[:3]])
        return f"""Kính gửi Quý khách,

Hồ sơ vay {app['loan_amount']//1_000_000}M VNĐ của Quý khách (Mã: {app['case_id']}) hiện cần xem xét thêm.

Lý do chính:
- Thời gian làm việc hiện tại ({app['employment_months']} tháng) chưa đáp ứng mức tối thiểu
  theo quy định cho khoản vay không có tài sản đảm bảo (theo TT39/2016, Điều 9).

Để được xem xét phê duyệt, Quý khách có thể:
{cf_lines}

Quý khách có thể liên hệ hotline 1800-LOANBOT hoặc nộp lại hồ sơ sau khi đáp ứng một trong các điều kiện trên.

Trân trọng,
FinTech Corp — LoanBot Credit Division
Decision Reference: {app['case_id']} | {datetime.now().strftime('%Y-%m-%d')}"""

# Test trên FC-2024-004
fc004 = {
    'case_id': 'FC-2024-004', 'credit_score': 645, 'dti_ratio': 0.42,
    'employment_months': 8, 'loan_amount': 400_000_000
}

explainer = LoanBotExplainer()

print('🔍 FEATURE IMPORTANCE (FC-2024-004):')
for feat in explainer.feature_importance(fc004):
    sign = '↑' if feat['direction'] == 'positive' else '↓'
    print(f"  {sign} {feat['label']}: {feat['impact']:+.2f}")

print('\n💡 COUNTERFACTUAL EXPLANATIONS (sorted by feasibility):')
cfs = explainer.counterfactual_explanation(fc004)
for cf in cfs:
    print(f"  [{cf.feasibility}] {cf.change_description}")
    print(f"         Impact: {cf.impact_score:.0%} chance of APPROVE if resolved")

print('\n📄 CUSTOMER LETTER:')
print('-' * 60)
print(explainer.generate_customer_letter(fc004, 'REVIEW', cfs))

## Section 5: Compliance Evidence Package Generator

Auto-generate NHNN audit evidence package với compliance status cho từng regulation article.

In [ ]:
# Section 5: Compliance Reporter

REGULATIONS = [
    ('EU AI Act', 'Art. 9',  'Risk Management System',           'DOCUMENT',       'explanation_coverage'),
    ('EU AI Act', 'Art. 10', 'Data Governance & Management',     'DATA_AUDIT',     'approval_rate_ratio_rural'),
    ('EU AI Act', 'Art. 11', 'Technical Documentation (Model Card)', 'MODEL_CARD', None),
    ('EU AI Act', 'Art. 12', 'Record-keeping & Automatic Logging', 'LOG_SYSTEM',   None),
    ('EU AI Act', 'Art. 14', 'Human Oversight Mechanisms',       'PROCESS',        None),
    ('EU AI Act', 'Art. 86', 'Right to Explanation',             'EXPLAINABILITY', 'explanation_coverage'),
    ('GDPR',      'Art. 22', 'Automated Decision Rights',        'PROCESS',        None),
    ('TT39/2016', 'Điều 9',  'Loan Purpose & Collateral Docs',   'PROCESS',        None),
    ('NHNN/2020', 'Điều 12', 'AI Risk Assessment Report',        'DOCUMENT',       None),
]

def generate_evidence_package(metrics: Dict[str, float], audit_period: str) -> dict:
    evidence_items = []
    non_compliant = []

    for reg, article, req, ev_type, metric_key in REGULATIONS:
        status = 'COMPLIANT'
        if metric_key:
            val = metrics.get(metric_key, None)
            if val is not None:
                if metric_key == 'explanation_coverage' and val < 1.0:
                    status = 'NON_COMPLIANT'
                elif metric_key == 'approval_rate_ratio_rural' and val < 0.80:
                    status = 'NON_COMPLIANT'

        item = {'regulation': reg, 'article': article, 'requirement': req,
                'evidence_type': ev_type, 'status': status,
                'evidence_path': f's3://loanbot-audit/{audit_period}/{ev_type.lower()}.json'}
        evidence_items.append(item)
        if status == 'NON_COMPLIANT':
            non_compliant.append(f'{reg} {article}')

    overall = 'NON_COMPLIANT' if non_compliant else 'COMPLIANT'
    return {'audit_period': audit_period, 'generated_at': datetime.now().isoformat(),
            'overall_status': overall, 'evidence_items': evidence_items,
            'non_compliant_articles': non_compliant, 'metrics': metrics}

# Generate package
metrics = compute_metrics(dataset)
package = generate_evidence_package(metrics, '2026-Q2')

print('📦 COMPLIANCE EVIDENCE PACKAGE — FinTech Corp | 2026-Q2')
print('=' * 75)
print(f'Overall Status: {"❌ NON_COMPLIANT" if package["overall_status"] == "NON_COMPLIANT" else "✅ COMPLIANT"}')
print()
print(f'{"Regulation":<12} {"Article":<10} {"Requirement":<38} {"Status":<15}')
print('-' * 75)
for item in package['evidence_items']:
    status_icon = '✅' if item['status'] == 'COMPLIANT' else '❌'
    print(f"{item['regulation']:<12} {item['article']:<10} {item['requirement'][:37]:<38} {status_icon} {item['status']}")
print('=' * 75)
if package['non_compliant_articles']:
    print(f'\n❌ Non-compliant articles: {", ".join(package["non_compliant_articles"])}')
    print('   Action required: Immediate remediation + 30-day corrective action plan')

## Section 6: Ethics Escalation Engine

Auto-determine escalation level based on bias severity, NHNN inquiry status, và loan value.

In [ ]:
# Section 6: Ethics Escalation Engine

class EscalationLevel(Enum):
    NONE         = 0
    LOG_ONLY     = 1
    ALERT        = 2
    HUMAN_REVIEW = 3
    ETHICS_BOARD = 4
    HALT         = 5

@dataclass
class EscalationDecision:
    level: EscalationLevel
    triggers: List[str]
    action: str
    timeline_hours: int
    escalate_to: str

class EthicsEscalationEngine:
    def evaluate(self, decision: dict, app: dict, fairness_metrics: dict,
                 nhnn_inquiry: bool = False) -> EscalationDecision:
        triggers = []
        max_level = EscalationLevel.NONE

        def update(level, trigger):
            nonlocal max_level
            triggers.append(trigger)
            if level.value > max_level.value:
                max_level = level

        # HALT: Severe disparate impact
        disparity = fairness_metrics.get('disparity_ratio', 1.0)
        if disparity < 0.60:
            update(EscalationLevel.HALT,
                   f'Severe disparate impact: ratio={disparity:.3f} (<0.60 minimum)')
        elif disparity < 0.80:
            update(EscalationLevel.ETHICS_BOARD,
                   f'Disparate impact violation: ratio={disparity:.3f} (<0.80 rule)')
        elif disparity < 0.85:
            update(EscalationLevel.ALERT,
                   f'Approaching 80% boundary: ratio={disparity:.3f}')

        # ETHICS_BOARD: Active NHNN inquiry
        if nhnn_inquiry:
            update(EscalationLevel.ETHICS_BOARD, 'Active NHNN regulatory inquiry in progress')

        # HUMAN_REVIEW: High-value + low confidence
        if app.get('loan_amount', 0) > 1_000_000_000 and decision.get('confidence', 1.0) < 0.65:
            update(EscalationLevel.HUMAN_REVIEW,
                   f"High-value loan {app['loan_amount']//1_000_000}M + confidence {decision['confidence']:.2f}")

        action_map = {
            EscalationLevel.NONE:         ('No action required', 0, 'AI System'),
            EscalationLevel.LOG_ONLY:     ('Log and monitor', 0, 'AI System'),
            EscalationLevel.ALERT:        ('Send alert — AI Team dashboard', 24, 'AI Product Manager'),
            EscalationLevel.HUMAN_REVIEW: ('Manual review by Credit Officer', 48, 'Credit Officer'),
            EscalationLevel.ETHICS_BOARD: ('Schedule Ethics Board review session', 72, 'Ethics Board Chair'),
            EscalationLevel.HALT:         ('HALT AI decisions in affected segment NOW', 1, 'CRO + CEO'),
        }
        action, timeline, escalate_to = action_map[max_level]
        return EscalationDecision(max_level, triggers, action, timeline, escalate_to)

# Test 3 scenarios
engine = EthicsEscalationEngine()

scenarios = [
    {'name': 'Scenario A: Mild disparity', 'decision': {'confidence': 0.72},
     'app': {'loan_amount': 200_000_000}, 'metrics': {'disparity_ratio': 0.83}, 'nhnn': False},
    {'name': 'Scenario B: Rural bias + NHNN inquiry', 'decision': {'confidence': 0.68},
     'app': {'loan_amount': 400_000_000}, 'metrics': {'disparity_ratio': 0.657}, 'nhnn': True},
    {'name': 'Scenario C: Severe bias — HALT', 'decision': {'confidence': 0.60},
     'app': {'loan_amount': 500_000_000}, 'metrics': {'disparity_ratio': 0.55}, 'nhnn': False},
]

print('⚖️  ETHICS ESCALATION ENGINE — Test Scenarios')
print('=' * 65)
for s in scenarios:
    result = engine.evaluate(s['decision'], s['app'], s['metrics'], s['nhnn'])
    print(f"\n{s['name']}")
    print(f"  Escalation Level: {result.level.name} (Level {result.level.value})")
    print(f"  Action: {result.action}")
    print(f"  Escalate to: {result.escalate_to} | Timeline: {result.timeline_hours}h")
    print(f"  Triggers: {', '.join(result.triggers)}")

## Section 7: GovernancePipeline — Full Integration

**TODO (bài tập):** Wire tất cả components vào một `GovernancePipeline` hoàn chỉnh.

Pipeline phải:
1. Accept một loan application
2. Run PolicyEngine để check violations
3. Run FairnessAuditor cho regional metrics
4. Determine EscalationLevel
5. Generate counterfactual explanation nếu decision != APPROVE
6. Generate compliance evidence package
7. Return `GovernanceResult` với tất cả artifacts

In [ ]:
# Section 7: GovernancePipeline — TODO

@dataclass
class GovernanceResult:
    application_id: str
    policy_violations: List[dict]
    fairness_violations: List[str]
    escalation: EscalationDecision
    explanation_letter: Optional[str]
    compliance_package: dict
    overall_governance_status: str  # 'PASS' / 'NEEDS_REVIEW' / 'CRITICAL_VIOLATION'

class GovernancePipeline:
    def __init__(self):
        self.policy_engine = None       # TODO: instantiate PolicyEngine
        self.fairness_auditor = None    # TODO: instantiate FairnessAuditor  
        self.explainer = LoanBotExplainer()
        self.escalation_engine = EthicsEscalationEngine()
        self.compliance_reporter = None  # TODO: instantiate ComplianceReporter

    def process(self, application: dict, decision: dict,
                all_cases: List[LoanCase]) -> GovernanceResult:
        """Full governance check for a loan decision."""
        # TODO Step 1: Run PolicyEngine on all_cases
        policy_violations = []

        # TODO Step 2: Run FairnessAuditor — get regional metrics
        fairness_violations = []

        # TODO Step 3: Compute fairness metrics for escalation
        # Compute disparity_ratio for application's region
        fairness_metrics = {'disparity_ratio': 1.0}  # TODO: compute real metric

        # TODO Step 4: Run EscalationEngine
        escalation = None  # TODO

        # TODO Step 5: Generate explanation if decision != APPROVE
        explanation = None
        if decision.get('outcome') != 'APPROVE':
            pass  # TODO: generate counterfactual + letter

        # TODO Step 6: Generate compliance package
        compliance = {}  # TODO

        # TODO Step 7: Determine overall governance status
        overall_status = 'PASS'  # TODO

        return GovernanceResult(
            application_id=application.get('case_id', 'UNKNOWN'),
            policy_violations=policy_violations,
            fairness_violations=fairness_violations,
            escalation=escalation,
            explanation_letter=explanation,
            compliance_package=compliance,
            overall_governance_status=overall_status
        )

print('📝 GovernancePipeline skeleton loaded.')
print('TODO: Complete the 7-step implementation above.')
print('\nTest case:')
print('  Application: FC-2024-004 (score=645, DTI=0.42, emp=8mo, region=rural)')
print('  Decision: REVIEW, confidence=0.73')
print('  Expected: Policy violations + Rural fairness violation + ETHICS_BOARD escalation')

In [ ]:
# SOLUTION (run after attempting TODO above)

class GovernancePipelineSolution:
    def __init__(self):
        self.explainer = LoanBotExplainer()
        self.escalation_engine = EthicsEscalationEngine()

    def process(self, application: dict, decision: dict,
                all_cases: List[LoanCase]) -> GovernanceResult:
        # Step 1: Policy violations
        policy_violations = run_policy_engine.__wrapped__(all_cases) if hasattr(run_policy_engine, '__wrapped__') else []
        metrics = compute_metrics(all_cases)
        policy_violations_list = []
        for policy in LOANBOT_CHARTER:
            val = metrics.get(policy.metric)
            if val is None: continue
            violated = (policy.compare == 'lt' and val < policy.threshold) or \
                       (policy.compare == 'gt' and val > policy.threshold)
            if violated:
                policy_violations_list.append({'policy_id': policy.policy_id,
                    'risk': policy.risk_level.name, 'value': val})

        # Step 2: Fairness violations
        fairness_violations = full_fairness_audit.__wrapped__(all_cases) if hasattr(full_fairness_audit, '__wrapped__') else []
        urban = [c for c in all_cases if c.region == 'urban']
        rural = [c for c in all_cases if c.region == 'rural']
        def ar(lst): return sum(1 for c in lst if c.agent_decision == 'APPROVE') / len(lst) if lst else 0
        disparity_ratio = ar(rural) / ar(urban) if ar(urban) > 0 else 0
        fairness_viol = ['Rural/Urban disparity'] if disparity_ratio < 0.80 else []

        # Step 3: Escalation
        fairness_metrics = {'disparity_ratio': disparity_ratio}
        nhnn_inquiry = len(policy_violations_list) > 1  # Simulate: 2+ violations → regulatory attention
        escalation = self.escalation_engine.evaluate(decision, application, fairness_metrics, nhnn_inquiry)

        # Step 4: Explanation
        explanation = None
        if decision.get('outcome', 'REVIEW') != 'APPROVE':
            cfs = self.explainer.counterfactual_explanation(application)
            explanation = self.explainer.generate_customer_letter(application, decision.get('outcome', 'REVIEW'), cfs)

        # Step 5: Compliance package
        compliance = generate_evidence_package(metrics, '2026-Q2')

        # Step 6: Overall status
        has_critical = any(v['risk'] == 'CRITICAL' for v in policy_violations_list)
        overall = 'CRITICAL_VIOLATION' if has_critical else ('NEEDS_REVIEW' if policy_violations_list else 'PASS')

        return GovernanceResult(
            application_id=application.get('case_id', 'UNKNOWN'),
            policy_violations=policy_violations_list,
            fairness_violations=fairness_viol,
            escalation=escalation,
            explanation_letter=explanation,
            compliance_package=compliance,
            overall_governance_status=overall
        )

# Run solution
pipeline = GovernancePipelineSolution()
result = pipeline.process(
    application={'case_id': 'FC-2024-004', 'credit_score': 645, 'dti_ratio': 0.42,
                 'employment_months': 8, 'loan_amount': 400_000_000},
    decision={'outcome': 'REVIEW', 'confidence': 0.73},
    all_cases=dataset
)

print('✅ GOVERNANCE PIPELINE RESULT — FC-2024-004')
print('=' * 55)
print(f'Overall Governance Status: {result.overall_governance_status}')
print(f'Policy Violations: {len(result.policy_violations)} found')
for v in result.policy_violations: print(f'  - {v["policy_id"]}: {v["risk"]} risk')
print(f'Fairness Violations: {result.fairness_violations}')
print(f'Escalation Level: {result.escalation.level.name if result.escalation else "N/A"}')
print(f'Escalate to: {result.escalation.escalate_to if result.escalation else "N/A"}')
print(f'Explanation Generated: {"Yes" if result.explanation_letter else "No"}')
print(f'Compliance Status: {result.compliance_package["overall_status"]}')
print()
print('📄 Customer Explanation Letter (first 200 chars):')
print(result.explanation_letter[:250] if result.explanation_letter else 'None')